In [86]:
import pandas as pd
import random

# Define possible values
genders = ['Male', 'Female']
occupations = ['Engineer', 'Doctor', 'Teacher', 'Artist', 'Scientist', 'Lawyer', 'Designer', 'Manager', 'Developer']

# Generate 10,000 rows
num_rows = 10000
data = []

for _ in range(num_rows):
    user_id = random.randint(1, 1000)  # Assume 1000 users
    profile_id = random.randint(1001, 2000)  # Assume 1000 profiles
    gender = random.choice(genders)
    occupation = random.choice(occupations)
    interaction = random.randint(1, 5)  # 1 to 5 interaction score
    data.append([user_id, profile_id, gender, occupation, interaction])

# Create DataFrame
df = pd.DataFrame(data, columns=['user_id', 'profile_id', 'gender', 'occupation', 'interaction'])

# Save to CSV
file_path = "/content/sample_data/matrimony_dataset.csv"
df.to_csv(file_path, index=False)

file_path


'/content/sample_data/matrimony_dataset.csv'

 1.2 Encode Categorical Features
You'll need to convert strings to numbers using LabelEncoder from sklearn

In [87]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import pickle

df = pd.read_csv("/content/sample_data/matrimony_dataset.csv")

# Encode gender and occupation
gender_encoder = LabelEncoder()
df['gender'] = gender_encoder.fit_transform(df['gender'].astype(str))

occupation_encoder = LabelEncoder()
df['occupation'] = occupation_encoder.fit_transform(df['occupation'].astype(str))

# Ab updated columns ko numpy arrays mein le lo:
#genders = df['gender'].values.astype(np.int32)
#occupations = df['occupation'].values.astype(np.int32)

# Convert to numpy arrays (int)
user_ids = df['user_id'].values.astype(np.int32)
profile_ids = df['profile_id'].values.astype(np.int32)
genders = df['gender'].values.astype(np.int32)
occupations = df['occupation'].values.astype(np.int32)

print(user_ids.dtype)       # int32
print(profile_ids.dtype)    # int32
print(genders.dtype)        # int32
print(occupations.dtype)    # int32

# Save encoders for later use in Flask
with open('gender_encoder.pkl', 'wb') as f:
    pickle.dump(gender_encoder, f)

with open('occupation_encoder.pkl', 'wb') as f:
    pickle.dump(occupation_encoder, f)


int32
int32
int32
int32


🔹 1.3 Split Data

In [88]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)


🧾 Code: Model Training (with Explanation)
✅ 2.1 Load and Prepare Data



In [89]:
import pandas as pd
import tensorflow as tf
import numpy as np
import pickle

# Load preprocessed data
df = pd.read_csv("/content/sample_data/matrimony_dataset.csv")

# Load encoded label mappings (if not already loaded)
with open('gender_encoder.pkl', 'rb') as f:
    gender_encoder = pickle.load(f)
with open('occupation_encoder.pkl', 'rb') as f:
    occupation_encoder = pickle.load(f)

# Split input features and target
user_ids = df['user_id'].values
profile_ids = df['profile_id'].values
genders = df['gender'].values
occupations = df['occupation'].values
interactions = df['interaction'].values


✅ 2.2 Build the Model with Embedding Layers

In [90]:
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense
from tensorflow.keras.models import Model

# Convert object columns to categorical codes
df['gender'] = df['gender'].astype('category')
df['occupation'] = df['occupation'].astype('category')

df['gender_code'] = df['gender'].cat.codes
df['occupation_code'] = df['occupation'].cat.codes

# Define constants using max values (safe for Embedding input_dim)
num_users = df['user_id'].max() + 1
num_profiles = df['profile_id'].max() + 1
num_genders = df['gender_code'].max() + 1
num_occupations = df['occupation_code'].max() + 1

embedding_dim = 32

# Inputs
user_input = Input(shape=(1,), name='user_input')
profile_input = Input(shape=(1,), name='profile_input')
gender_input = Input(shape=(1,), name='gender_input')
occupation_input = Input(shape=(1,), name='occupation_input')

# Embedding layers with correct input_dim
user_emb = Embedding(input_dim=num_users, output_dim=embedding_dim)(user_input)
profile_emb = Embedding(input_dim=num_profiles, output_dim=embedding_dim)(profile_input)
gender_emb = Embedding(input_dim=num_genders, output_dim=4)(gender_input)
occupation_emb = Embedding(input_dim=num_occupations, output_dim=4)(occupation_input)

# Flatten embeddings
user_vec = Flatten()(user_emb)
profile_vec = Flatten()(profile_emb)
gender_vec = Flatten()(gender_emb)
occupation_vec = Flatten()(occupation_emb)

# Concatenate all vectors
merged = Concatenate()([user_vec, profile_vec, gender_vec, occupation_vec])

# Dense layers
x = Dense(64, activation='relu')(merged)
x = Dense(32, activation='relu')(x)
output = Dense(1)(x)  # Interaction score

# Define and compile model
model = Model(inputs=[user_input, profile_input, gender_input, occupation_input], outputs=output)
model.compile(optimizer='adam', loss='mse')
model.summary()


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ profile_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender_input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ occupation_input    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_32        │ (None, 1, 32)     │     32,032 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_33        │ (None, 1, 32)     │     64,032 │ profile_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_34        │ (None, 1, 4)      │          8 │ gender_input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_35        │ (None, 1, 4)      │         36 │ occupation_input… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_32          │ (None, 32)        │          0 │ embedding_32[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_33          │ (None, 32)        │          0 │ embedding_33[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_34          │ (None, 4)         │          0 │ embedding_34[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_35          │ (None, 4)         │          0 │ embedding_35[0][… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 72)        │          0 │ flatten_32[0][0], │
│ (Concatenate)       │                   │            │ flatten_33[0][0], │
│                     │                   │            │ flatten_34[0][0], │
│                     │                   │            │ flatten_35[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 64)        │      4,672 │ concatenate_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 32)        │      2,080 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 1)         │         33 │ dense_25[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 102,893 (401.93 KB)

 Trainable params: 102,893 (401.93 KB)

 Non-trainable params: 0 (0.00 B)

In [91]:
print(user_ids.dtype)
print(profile_ids.dtype)
print(genders.dtype)
print(occupations.dtype)
print(interactions.dtype)

int64
int64
object
object
int64


In [92]:
import numpy as np

user_ids = df['user_id'].values.astype(np.int32)
profile_ids = df['profile_id'].values.astype(np.int32)
genders = df['gender'].astype('category').cat.codes.values.astype(np.int32)
occupations = df['occupation'].astype('category').cat.codes.values.astype(np.int32)
interactions = df['interaction'].values.astype(np.float32)


In [93]:
print(user_ids.dtype)
print(profile_ids.dtype)
print(genders.dtype)
print(occupations.dtype)
print(interactions.dtype)

int32
int32
int32
int32
float32


✅ 2.3 Train the Model

In [94]:
model.fit(
    x=[user_ids, profile_ids, genders, occupations],
    y=interactions,
    batch_size=64,
    epochs=10,
    validation_split=0.2
)



Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 7.4216 - val_loss: 2.0830
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.8706 - val_loss: 2.1376
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.7084 - val_loss: 2.2046
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.6304 - val_loss: 2.2353
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.6139 - val_loss: 2.2928
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.5990 - val_loss: 2.2965
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.5881 - val_loss: 2.2779
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.5380 - val_loss: 2.2897
Epoch 9/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.5250 - val_loss: 2.2734
Epoch 10/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.4907 - val_loss: 2.4002


✅ Step 1: Save the Model and Mappings

Save the trained model and any mappings (gender, occupation, etc.) used during encoding:

In [95]:
# Save model
model.save('matrimony_recommender_model.h5')

# Save mappings used for gender and occupation
import pickle

gender_mapping = dict(enumerate(df['gender'].cat.categories))
occupation_mapping = dict(enumerate(df['occupation'].cat.categories))

with open('gender_mapping.pkl', 'wb') as f:
    pickle.dump(gender_mapping, f)

with open('occupation_mapping.pkl', 'wb') as f:
    pickle.dump(occupation_mapping, f)


In [99]:
print(df.columns)


Index(['user_id', 'profile_id', 'gender', 'occupation', 'interaction',
       'gender_code', 'occupation_code'],
      dtype='object')


✅ Step 2: Prepare JSON Data for Profile Lookup

If you want to return full profile details with recommendations:

In [100]:
# Create a dictionary of profiles
profile_data = df.drop_duplicates(subset='profile_id')[
    ['profile_id', 'gender', 'occupation']
].to_dict(orient='records')

import json
with open('data1.json', 'w') as f:
    json.dump(profile_data, f)


✅ Step 3: Load Model and Serve Recommendations in Flask

Here's a basic structure:

In [6]:
!pip install flask pyngrok --quiet

from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
import numpy as np
import pickle
import json
from pyngrok import ngrok
import threading

app = Flask(__name__)

# Load model and mappings
model = load_model('matrimony_recommender_model.h5', compile=False)
gender_mapping = pickle.load(open('gender_mapping.pkl', 'rb'))
occupation_mapping = pickle.load(open('occupation_mapping.pkl', 'rb'))

# Invert mappings
gender_inv_map = {v: k for k, v in gender_mapping.items()}
occupation_inv_map = {v: k for k, v in occupation_mapping.items()}

# Load profile data
with open('data1.json') as f:
    profile_data = json.load(f)

@app.route('/recommend', methods=['POST'])
def recommend():
    data = request.json
    user_id = int(data['user_id'])
    user_gender = data['gender']
    user_occupation = data['occupation']

    gender_code = gender_inv_map.get(user_gender, 0)
    occupation_code = occupation_inv_map.get(user_occupation, 0)

    profile_ids = np.array([p['profile_id'] for p in profile_data])
    user_ids = np.full_like(profile_ids, user_id)
    genders = np.full_like(profile_ids, gender_code)
    occupations = np.full_like(profile_ids, occupation_code)

    # Predict interaction scores
    predictions = model.predict([user_ids, profile_ids, genders, occupations])
    top_indices = predictions.flatten().argsort()[-10:][::-1]

    top_profiles = [profile_data[i] for i in top_indices]
    for i, profile in enumerate(top_profiles):
        profile['match_score'] = float(predictions[top_indices[i]])

    return jsonify(top_profiles)

# Function to run Flask app
def run_app():
    app.run(host='0.0.0.0', port=5000)

# Start ngrok tunnel
ngrok.set_auth_token("2xDJMlceMxWUEBQm2yVuVG3W01u_7t3Rph4uezCQSbXSgcSBp")  # Optional: Only if using auth
public_url = ngrok.connect(5000)
print("🚀 Ngrok public URL:", public_url)

# Start Flask app in a separate thread
thread = threading.Thread(target=run_app)
thread.start()


🚀 Ngrok public URL: NgrokTunnel: "https://e9e9-35-231-120-55.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


In [3]:
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
import numpy as np
import pickle
import json

app = Flask(__name__)

# Load model and mappings
model = load_model('matrimony_recommender_model.h5', compile=False)
gender_mapping = pickle.load(open('gender_mapping.pkl', 'rb'))
occupation_mapping = pickle.load(open('occupation_mapping.pkl', 'rb'))

# Invert mappings
gender_inv_map = {v: k for k, v in gender_mapping.items()}
occupation_inv_map = {v: k for k, v in occupation_mapping.items()}

# Load profile data
with open('data1.json') as f:
    profile_data = json.load(f)

@app.route('/recommend', methods=['POST'])
def recommend():
    data = request.json
    user_id = int(data['user_id'])
    user_gender = data['gender']
    user_occupation = data['occupation']

    gender_code = gender_inv_map.get(user_gender, 0)
    occupation_code = occupation_inv_map.get(user_occupation, 0)

    profile_ids = np.array([p['profile_id'] for p in profile_data])
    user_ids = np.full_like(profile_ids, user_id)
    genders = np.full_like(profile_ids, gender_code)
    occupations = np.full_like(profile_ids, occupation_code)

    # Predict interaction scores
    predictions = model.predict([user_ids, profile_ids, genders, occupations])
    top_indices = predictions.flatten().argsort()[-10:][::-1]

    top_profiles = [profile_data[i] for i in top_indices]
    for i, profile in enumerate(top_profiles):
        profile['match_score'] = float(predictions[top_indices[i]])

    return jsonify(top_profiles)

if __name__ == '__main__':
     app.run(host='0.0.0.0', port=5000, debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
